# Week 3: IMPROVED Hybrid Classifier Validation

**Fixes from first attempt:**
1. **Increased margin: 0.05 → 0.20** (more conservative)
2. **Improved language prototype** (stronger documentation words)
3. **Added fallback layer** (only classify if similarity > threshold)
4. **Target: Code coverage 10-20% (not 35%!)**

---

## 1-7. Setup (Same as Before)

Run cells 1-7 from the previous notebook (installation, imports, model loading, embedding model, entropy functions)

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate sentence-transformers scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as scipy_entropy
import time
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports successful")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print(f"✅ Model loaded on {model.device}")
print(f"   Vocabulary size: {len(tokenizer):,}")

In [ ]:
# Cell 4: Embedding Model
print("Loading sentence-transformers...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

In [ ]:
# Cell 5: Entropy Functions
def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def shannon_entropy(logits: np.ndarray) -> float:
    probs = softmax(logits)
    return float(scipy_entropy(probs, base=2))

def get_top_k_predictions(logits: np.ndarray, k: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    probs = softmax(logits)
    top_k_indices = np.argsort(logits)[-k:][::-1]
    top_k_probs = probs[top_k_indices]
    return top_k_indices, top_k_probs

print("✅ Entropy functions defined")

## 6. IMPROVED: Conservative Keyword Sets

In [ ]:
# CODE KEYWORDS - More conservative, only clear programming terms
CODE_KEYWORDS = {
    # Core Python
    'if', 'else', 'elif', 'for', 'while', 'break', 'continue', 'pass',
    'return', 'yield', 'raise', 'try', 'except', 'finally', 'with', 'as',
    'def', 'class', 'lambda', 'async', 'await',
    'import', 'from',
    
    # Core JavaScript
    'function', 'const', 'let', 'var', 'switch', 'case', 'default',
    'export', 'require', 'module',
    
    # ONLY VERY SPECIFIC domain terms (not too many!)
    'pandas', 'numpy', 'pd', 'np',
    'requests', 'flask', 'django', 'fastapi', 'FastAPI',
    'React', 'useState', 'useEffect', 'useContext',
    'firebase', 'Firebase', 'auth',
    'DataFrame', 'read_csv',
}

# LANGUAGE KEYWORDS - Expanded with more documentation terms
LANGUAGE_WORDS = {
    # Question words
    'what', 'how', 'why', 'when', 'where', 'which', 'who',
    
    # Strong documentation verbs
    'explain', 'describe', 'summarize', 'show', 'tell',
    'write', 'create', 'add', 'update', 'implement',
    
    # Very common English
    'the', 'a', 'an', 'this', 'that', 'these', 'those',
    'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did',
    'will', 'would', 'should', 'can', 'could',
    'to', 'of', 'for', 'with', 'at', 'by',
    'I', 'you', 'it', 'we', 'they', 'me', 'my',
    
    # Descriptive words
    'function', 'method', 'code', 'example', 'using',
}

CODE_OPERATORS = {'+', '-', '*', '/', '=', '==', '!=', '<', '>', '{', '}', '[', ']', '(', ')', ';', ':', ','}

CODE_KEYWORDS_LOWER = {k.lower() for k in CODE_KEYWORDS}
LANGUAGE_WORDS_LOWER = {w.lower() for w in LANGUAGE_WORDS}

print(f"✅ Conservative keyword sets:")
print(f"   Code keywords: {len(CODE_KEYWORDS)} (reduced from 300)")
print(f"   Language words: {len(LANGUAGE_WORDS)}")

In [ ]:
# Keyword-only classifier (same)
def classify_token_keyword_only(token: str) -> str:
    token_clean = token.strip().lower()
    if token.strip() in CODE_OPERATORS:
        return 'code'
    if token_clean in CODE_KEYWORDS_LOWER:
        return 'code'
    if token_clean in LANGUAGE_WORDS_LOWER:
        return 'language'
    return 'other'

print("✅ Keyword-only classifier defined")

## 7. IMPROVED: Conservative Hybrid Classifier

**Key Changes:**
1. **Margin increased: 0.05 → 0.20** (4x more conservative)
2. **Minimum similarity threshold: 0.5** (must be clearly similar)
3. **Stronger language prototype** with documentation examples

In [ ]:
print("Building IMPROVED prototypes...")

# Code prototype: Focus on CLEAR programming constructs
code_prototype_examples = [
    # Pure programming constructs (60%)
    'function', 'class', 'import', 'return', 'if', 'else', 'for', 'while',
    'const', 'let', 'var', 'def', 'async', 'await', 'try', 'catch',
    'public', 'private', 'static', 'void', 'int', 'string', 'array',
    'print', 'console', 'log', 'typeof', 'null', 'true', 'false',
    
    # Domain-specific (40%) - REDUCED from 50%
    'pandas', 'numpy', 'requests', 'firebase', 'react',
    'useState', 'FastAPI', 'axios', 'django',
    'DataFrame', 'read_csv', 'fetch', 'router',
]

code_embeddings = embedding_model.encode(code_prototype_examples)
code_prototype = np.mean(code_embeddings, axis=0).reshape(1, -1)

# Language prototype: STRONGER documentation focus
language_prototype_examples = [
    # Documentation/explanation words (70% - INCREASED)
    'explain', 'describe', 'summarize', 'documentation', 'comment',
    'write', 'create', 'add', 'update', 'implement', 'show', 'tell',
    'example', 'tutorial', 'guide', 'instruction', 'demonstration',
    'overview', 'summary', 'description', 'explanation',
    
    # Question words
    'how', 'what', 'why', 'when', 'where', 'which',
    
    # Common language
    'the', 'this', 'that', 'these', 'those', 'a', 'an',
    'is', 'are', 'was', 'were', 'have', 'has', 'had',
    'should', 'would', 'could', 'will', 'can',
]

language_embeddings = embedding_model.encode(language_prototype_examples)
language_prototype = np.mean(language_embeddings, axis=0).reshape(1, -1)

embedding_cache = {}

def classify_token_hybrid_improved(token: str, margin: float = 0.20, min_similarity: float = 0.5) -> str:
    """
    IMPROVED: More conservative hybrid classifier.
    
    Changes:
    - margin: 0.05 → 0.20 (4x more conservative)
    - min_similarity: 0.5 (must be clearly similar to a prototype)
    """
    # Stage 1: Keyword lookup
    keyword_result = classify_token_keyword_only(token)
    if keyword_result != 'other':
        return keyword_result
    
    # Stage 2: Conservative embedding similarity
    if token not in embedding_cache:
        embedding_cache[token] = embedding_model.encode([token])[0].reshape(1, -1)
    
    token_emb = embedding_cache[token]
    
    sim_code = cosine_similarity(token_emb, code_prototype)[0][0]
    sim_lang = cosine_similarity(token_emb, language_prototype)[0][0]
    
    # Check minimum similarity threshold
    max_sim = max(sim_code, sim_lang)
    if max_sim < min_similarity:
        return 'other'  # Not clearly similar to either prototype
    
    # Classification with LARGER margin
    diff = sim_code - sim_lang
    
    if diff > margin:
        return 'code'
    elif diff < -margin:
        return 'language'
    else:
        return 'other'  # Ambiguous

print("✅ IMPROVED hybrid classifier defined")
print(f"   Code prototype: {len(code_prototype_examples)} examples (40% domain-specific)")
print(f"   Language prototype: {len(language_prototype_examples)} examples (70% documentation)")
print(f"   Margin: 0.20 (was 0.05)")
print(f"   Min similarity: 0.5")

## 8-15. Run Experiments (Same as Before)

Use the same experiment pipeline from cells 8-15 of the original notebook.

In [ ]:
# Cell 8: CCE Computation
def compute_cce_fast(logits: np.ndarray, vocab_classifications: Dict, return_details: bool = False) -> Dict:
    vocab_size = len(logits)
    probs = softmax(logits)
    
    code_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'code']
    language_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'language']
    other_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'other']
    
    h_total = shannon_entropy(logits)
    h_code = shannon_entropy(logits[code_indices]) if len(code_indices) > 0 else 0.0
    h_language = shannon_entropy(logits[language_indices]) if len(language_indices) > 0 else 0.0
    cce = h_code - h_language
    
    code_prob_mass = np.sum(probs[code_indices]) if len(code_indices) > 0 else 0.0
    language_prob_mass = np.sum(probs[language_indices]) if len(language_indices) > 0 else 0.0
    other_prob_mass = np.sum(probs[other_indices]) if len(other_indices) > 0 else 0.0
    
    result = {
        'contrastive_entropy': float(cce),
        'code_entropy': float(h_code),
        'language_entropy': float(h_language),
        'total_entropy': float(h_total),
        'code_prob_mass': float(code_prob_mass),
        'language_prob_mass': float(language_prob_mass),
        'other_prob_mass': float(other_prob_mass),
    }
    
    if return_details:
        result['coverage'] = {
            'code_count': len(code_indices),
            'language_count': len(language_indices),
            'other_count': len(other_indices),
            'code_pct': len(code_indices) / vocab_size,
            'language_pct': len(language_indices) / vocab_size,
            'other_pct': len(other_indices) / vocab_size,
        }
    
    return result

print("✅ CCE computation defined")

In [ ]:
# Cell 9: Test Examples
TEST_EXAMPLES = [
    {'id': 'code_1', 'type': 'missing_context', 'prompt': 'How do I use the requests library to make an HTTP GET request in Python? Show me the code.'},
    {'id': 'code_2', 'type': 'missing_context', 'prompt': 'Write a React component that uses useState. Show the import and component.'},
    {'id': 'code_3', 'type': 'missing_context', 'prompt': 'How do I authenticate with Firebase in a TypeScript project? Show the auth code.'},
    {'id': 'code_4', 'type': 'missing_context', 'prompt': 'Write a function that uses pandas to read a CSV file and filter rows.'},
    {'id': 'code_5', 'type': 'missing_context', 'prompt': 'How do I create a FastAPI endpoint that handles POST requests with JSON body?'},
    {'id': 'lang_1', 'type': 'language_choice', 'prompt': 'Explain what this function does: def add(a, b): return a + b'},
    {'id': 'lang_2', 'type': 'language_choice', 'prompt': 'Write a comment describing this code: for item in items: process(item)'},
    {'id': 'lang_3', 'type': 'language_choice', 'prompt': 'Summarize this code: class User: def __init__(self, name): self.name = name'},
    {'id': 'lang_4', 'type': 'language_choice', 'prompt': 'Add a docstring to: def multiply(x, y): return x * y'},
    {'id': 'lang_5', 'type': 'language_choice', 'prompt': 'Describe this function: def is_even(n): return n % 2 == 0'},
]

print(f"✅ {len(TEST_EXAMPLES)} test examples loaded")

In [ ]:
# Cell 10: Pre-classify vocabulary
def classify_vocabulary_cached(classifier_func, vocab_size):
    print(f"Pre-classifying vocabulary ({vocab_size:,} tokens)...")
    classifications = {}
    for token_id in tqdm(range(vocab_size), desc="Classifying vocab"):
        token_str = tokenizer.decode([token_id])
        classifications[token_id] = classifier_func(token_str)
    return classifications

vocab_size = len(tokenizer)

print("\n1/2 Classifying with keyword-only...")
vocab_classifications_keyword = classify_vocabulary_cached(classify_token_keyword_only, vocab_size)

print("\n2/2 Classifying with IMPROVED hybrid...")
vocab_classifications_hybrid = classify_vocabulary_cached(classify_token_hybrid_improved, vocab_size)

# Show coverage
code_count_kw = sum(1 for c in vocab_classifications_keyword.values() if c == 'code')
lang_count_kw = sum(1 for c in vocab_classifications_keyword.values() if c == 'language')
other_count_kw = sum(1 for c in vocab_classifications_keyword.values() if c == 'other')

code_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'code')
lang_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'language')
other_count_hy = sum(1 for c in vocab_classifications_hybrid.values() if c == 'other')

print("\n" + "="*60)
print("Vocabulary Classification Results:")
print("="*60)
print(f"Keyword:  {code_count_kw:5,} code | {lang_count_kw:5,} lang | {other_count_kw:5,} other ({code_count_kw/vocab_size:5.1%} code)")
print(f"Hybrid:   {code_count_hy:5,} code | {lang_count_hy:5,} lang | {other_count_hy:5,} other ({code_count_hy/vocab_size:5.1%} code)")
print(f"\n⚠️  Target: Code coverage should be 10-20% (not 35%!)")
print(f"   Actual: {code_count_hy/vocab_size:.1%}")

In [ ]:
# Cell 11: Run experiments
def run_experiment_fast(example: Dict, vocab_classifications: Dict, method_name: str) -> Dict:
    prompt = example['prompt']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    first_logits = outputs.scores[0][0].cpu().numpy()
    generated = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    cce_result = compute_cce_fast(first_logits, vocab_classifications, return_details=True)
    top_k_indices, top_k_probs = get_top_k_predictions(first_logits, k=10)
    top_k_tokens = [tokenizer.decode([idx]) for idx in top_k_indices]
    top_k_predictions = list(zip(top_k_tokens, top_k_probs))
    
    return {
        'id': example['id'],
        'type': example['type'],
        'method': method_name,
        'prompt': prompt,
        'generated_text': generated[len(prompt):],
        **cce_result,
        'top_k_predictions': top_k_predictions,
    }

results = []
for example in tqdm(TEST_EXAMPLES, desc="Processing examples"):
    result_keyword = run_experiment_fast(example, vocab_classifications_keyword, 'keyword')
    results.append(result_keyword)
    
    result_hybrid = run_experiment_fast(example, vocab_classifications_hybrid, 'hybrid')
    results.append(result_hybrid)
    
    print(f"\n{example['id']}:")
    print(f"  Keyword CCE: {result_keyword['contrastive_entropy']:+.3f}")
    print(f"  Hybrid  CCE: {result_hybrid['contrastive_entropy']:+.3f}")

print(f"\n✅ Completed {len(results)} experiments")

In [ ]:
# Cell 12: Analysis
df = pd.DataFrame(results)
df_keyword = df[df['method'] == 'keyword'].copy()
df_hybrid = df[df['method'] == 'hybrid'].copy()

missing_cces = df_hybrid[df_hybrid['type'] == 'missing_context']['contrastive_entropy'].values
language_cces = df_hybrid[df_hybrid['type'] == 'language_choice']['contrastive_entropy'].values

from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(missing_cces, language_cces)

mean_diff = missing_cces.mean() - language_cces.mean()
pooled_std = np.sqrt(((len(missing_cces)-1)*missing_cces.std()**2 + 
                      (len(language_cces)-1)*language_cces.std()**2) / 
                     (len(missing_cces) + len(language_cces) - 2))
cohens_d = mean_diff / pooled_std

print("="*60)
print("HYBRID METHOD RESULTS")
print("="*60)
print(f"Missing context CCE: {missing_cces.mean():+.3f}")
print(f"Language choice CCE: {language_cces.mean():+.3f}")
print(f"Separation: {mean_diff:+.3f}")
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"Cohen's d: {cohens_d:.3f}")
print(f"\nHypothesis supported: {'YES ✅' if p_value < 0.05 else 'NO ❌'}")

# Save results
df.to_csv('week3_improved_results.csv', index=False)
print("\n✅ Results saved to week3_improved_results.csv")